In [0]:
%run "../common/config"

In [0]:


# COMMAND ----------
import requests
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

schema = StructType([
    StructField('schemeCode', StringType(), True),
    StructField('schemeName', StringType(), True),
    StructField('isinGrowth', StringType(), True),
    StructField('fundHouse', StringType(), True),
    StructField('schemeCategory', StringType(), True),
    StructField('date', StringType(), True),
    StructField('nav', StringType(), True),
    StructField('schemeType', StringType(), True),
    StructField('isinDivReinvestment', StringType(), True),
])

schemes = requests.get(f"{MFAPI_BASE_URL}/latest").json()
latest_date = F.date_sub(F.current_date(), 30)

schemes_df = spark.createDataFrame(schemes, schema=schema)
schemes_df = (
    schemes_df
    .withColumn("date", F.to_date(F.col("date"), "dd-MM-yyyy"))
    .filter(F.col("date") > latest_date)
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .filter(~F.lower(F.col("schemeName")).rlike("regular|dividend|idcw|reinvestment"))
)

schemes_df.write.format("delta").mode("overwrite").saveAsTable(f"{BRONZE}.mf_scheme_master_raw")
display(schemes_df)

schemeCode,schemeName,isinGrowth,fundHouse,schemeCategory,date,nav,schemeType,isinDivReinvestment,ingestion_timestamp
100319,LIC MF Gilt Fund - Growth,INF767K01733,LIC Mutual Fund,Income/Debt Oriented Schemes - Gilt Fund,2026-08-31,34.79070,Open Ended Schemes,null,2026-09-01T11:15:36.734Z
101408,BARODA BNP PARIBAS LIQUID FUND - Growth Option,INF955L01575,Baroda BNP Paribas Mutual Fund,Debt Scheme - Liquid Fund,2026-08-31,4743.80780,Open Ended Schemes,null,2026-09-01T11:15:36.734Z
101705,ICICI Prudential BSE Sensex ETF,INF346A01034,ICICI Prudential Mutual Fund,Other Scheme - Other ETFs,2026-08-31,885.68730,Open Ended Schemes,null,2026-09-01T11:15:36.734Z
103490,Quantum Value Fund - Direct Plan - Growth Option,INF082J01036,Quantum Mutual Fund,Equity Scheme - Value Fund,2026-08-31,126.08000,Open Ended Schemes,null,2026-09-01T11:15:36.734Z
103734,Quantum Liquid Fund - Direct Plan - Growth Option,INF082J01127,Quantum Mutual Fund,Debt Scheme - Liquid Fund,2026-08-31,37.66360,Open Ended Schemes,null,2026-09-01T11:15:36.734Z
105463,UTI - Gold Exchange Traded Fund - Direct Plan - Growth,INF789F1AUX7,UTI Mutual Fund,Exchange Traded Funds (ETFs) - Gold ETF,2026-08-31,129.85740,Open Ended Schemes,null,2026-09-01T11:15:36.734Z
106193,KOTAK GOLD ETF,INF174KA1HJ8,Kotak Mahindra Mutual Fund,Other Scheme - Gold ETF,2026-08-31,128.77920,Open Ended Schemes,null,2026-09-01T11:15:36.734Z
106929,Kotak Nifty PSU Bank ETF,INF174KA1A86,Kotak Mahindra Mutual Fund,Exchange Traded Funds (ETFs) - Equity ETF,2026-08-31,86.21420,Open Ended Schemes,null,2026-09-01T11:15:36.734Z
107693,Quantum Gold ETF,INF082J01408,Quantum Mutual Fund,Other Scheme - Gold ETF,2026-08-31,127.85600,Open Ended Schemes,null,2026-09-01T11:15:36.734Z
108479,Quantum Nifty 50 ETF,INF082J01499,Quantum Mutual Fund,Other Scheme - Other ETFs,2026-08-31,265.27520,Open Ended Schemes,null,2026-09-01T11:15:36.734Z
